# 24 - Python 3.15 New Features


## 1. High-Frequency Statistical Sampling Profiler (`profiling.sampling` / `Tachyon`) (PEP 799)

Python 3.15 standart kütüphanesine eklenen `profiling.sampling` modülü (kod adı *Tachyon*), çalışan bir Python sürecine kod değişikliği veya yeniden başlatma gerektirmeden bağlanarak saniyede 1 milyon örneğe kadar çağrı yığını (call stack) ve CPU/wall-time profili çıkaran düşük maliyetli örnekleme profilleyicisidir.

**Syntax**

```bash
python -m profiling.sampling attach <PID>
python -m profiling.sampling run --flamegraph script.py
```

**Mimari Özellikler ve Çıktı Formatları**

* **Sıfıra Yakın Çalışma Maliyeti:** Yığın çerçevelerini (stack frames) doğrudan süreç belleğinden okuyarak (async-safe / non-intrusive) üretim ortamındaki servisleri yavaşlatmadan darboğazları tespit eder.
* **Yerleşik Görselleştirme ve Raporlama:** Konsol özet tablolarının yanı sıra doğrudan etkileşimli **Flamegraph** (HTML), **Firefox Profiler** JSON ve **Gecko** formatlarında çıktı üretebilir; ayrıca free-threaded (No-GIL) iş parçacıklarını ve `asyncio` görev yığınlarını ayrı ayrı izler.


In [ ]:
import os
import sys

print('Python version:', sys.version.split()[0])
print('Current process PID for attach:', os.getpid())

try:
    import profiling.sampling as sampling_profiler

    print('profiling.sampling module loaded:', sampling_profiler.__name__)
except ImportError:
    print('profiling.sampling requires Python 3.15+')


## 2. UTF-8 Mode Enabled by Default Across All Platforms (PEP 686)

Python 3.15 ile işletim sistemi yerel ayarlarından (Windows code page `cp1252`, `cp1254` vb.) bağımsız olarak UTF-8 modu varsayılan hale getirilmiş; `open()`, `Path.read_text()`, `Path.write_text()` ve standart giriş/çıkış akışları `encoding` belirtilmediğinde otomatik olarak UTF-8 kullanmaya başlamıştır.

**Syntax**

```Python
from pathlib import Path

Path('notlar.txt').write_text('Türkçe karakterler: ş, ğ, ü, ö, ç, İ')
content = Path('notlar.txt').read_text()
```

**Pratik Etkiler ve Geçiş Seçenekleri**

* **Platformlar Arası Tutarlılık:** Windows üzerinde unutulan `encoding='utf-8'` parametresi nedeniyle oluşan `UnicodeEncodeError` / `UnicodeDecodeError` hataları ortadan kalkar.
* **Yerel Kodlama Geri Dönüşü (`encoding='locale'`):** Eski sistem dosyalarıyla çalışırken işletim sisteminin yerel kodlamasını kullanmak gerektiğinde açıkça `encoding='locale'` parametresi veya `PYTHONUTF8=0` ortam değişkeni kullanılabilir.


In [ ]:
import locale
import sys
from tempfile import TemporaryDirectory
from pathlib import Path

print('UTF-8 mode active:', bool(sys.flags.utf8_mode))
print('Preferred encoding:', locale.getpreferredencoding(False))

with TemporaryDirectory() as tmp_dir:
    sample_file = Path(tmp_dir) / 'utf8_test.txt'
    sample_file.write_text('Flamenko akorları: Fmaj7, E7, Am — şğüöçİ')
    print('Read without explicit encoding:', sample_file.read_text())


## 3. Lazy Imports (`lazy import` & `lazy from ... import`) (PEP 810)

Python 3.15 ile sözdizimi düzeyinde desteklenen `lazy import` deyimi, belirtilen modülün `import` satırında hemen yüklenmesi yerine modül adı veya üyesi kod içinde ilk kez erişildiği anda yüklenmesini (deferred loading) sağlar.

**Syntax**

```python
lazy import json
lazy from pathlib import Path
```

**Çalışma Prensibi ve Dikkat Edilmesi Gerekenler**

* **Hızlı Başlangıç (Startup Time) ve Düşük Bellek:** CLI araçları, büyük veri/AI projeleri veya nadiren kullanılan alt komutlarda ağır kütüphaneler yalnızca ilgili kod dalı çalıştığında içe aktarılır; fonksiyon içine gömülü `import` yazma ihtiyacını ortadan kaldırır.
* **Hata Zamanlaması:** `lazy import` ile işaretlenen var olmayan bir modül `ModuleNotFoundError` hatasını `import` satırında değil, değişkene ilk erişim anında fırlatır; ayrıca `try ... except` blokları içinde veya `*` (`from x import *`) ile kullanılamaz.


In [ ]:
import sys

lazy import statistics
lazy from fractions import Fraction

print('statistics loaded before use:', 'statistics' in sys.modules)
print('fractions loaded before use:', 'fractions' in sys.modules)

mean_val: float = statistics.mean([10.0, 20.0, 30.0])
ratio: Fraction = Fraction(3, 4)

print('statistics loaded after use:', 'statistics' in sys.modules)
print('Mean:', mean_val, '| Fraction:', ratio)


## 4. `PyBytesWriter` C-API (PEP 782), Explicit `int`/Index Conversions & `math.integer` Updates

Python 3.15 ile C uzantıları için gereksiz bellek kopyalamalarını ortadan kaldıran `PyBytesWriter` API'si eklenmiş, `bytes()` ve `bytearray()` kurucularında örtük `__index__()` dönüşümü kullanım dışı bırakılmış ve tam sayı odaklı matematik yardımcıları genişletilmiştir.

**Syntax**

```python
# Explicit conversion required instead of relying on implicit __index__
raw = bytes(int(custom_index_obj))
```

**Önemli Dil ve Kütüphane İyileştirmeleri**

* **Sıfır Kopyalı `bytes` Oluşturma (PEP 782):** C uzantılarında boyutu önceden tam bilinmeyen ikili veriler `PyBytesWriter` tamponuna doğrudan yazılır ve ek bellek ayırma/kopyalama yapılmadan `PyBytesObject` nesnesine dönüştürülür (standart kütüphanedeki JSON, Base64 ve sıkıştırma modüllerini hızlandırır).
* **Örtük Tam Sayı Dönüşümünün Kaldırılması:** `bytes(obj)` çağrısına tam sayı yerine yalnızca `__index__()` tanımlı özel bir nesne verildiğinde ortaya çıkan belirsiz davranış `DeprecationWarning` ile işaretlenmiştir.
* **Gelişmiş `frozendict` ve JIT İyileştirmeleri:** CPython JIT derleyicisi özellikle macOS AArch64 ve x86-64 platformlarında önemli hız kazanımları sağlarken, standart kütüphane genelinde tip güvenliği ve bellek ayak izi iyileştirilmiştir.


In [ ]:
class SampleIndex:
    def __init__(self, size: int) -> None:
        self.size = size

    def __index__(self) -> int:
        return self.size


idx = SampleIndex(4)
buffer: bytes = bytes(int(idx))
print('Explicitly sized zero buffer:', buffer, '| Length:', len(buffer))


## 5. Dedicated `math.integer` Module & Improved Error Diagnostics

Python 3.15 ile `math` modülündeki saf tam sayı (arbitrary-precision integer) işlemleri için ayrılmış `math.integer` alt modülü düzenlenmiş, ayrıca sözdizimi ve tip hatalarında geliştiriciye doğrudan neden-sonuç gösteren hata mesajları genişletilmiştir.

**Syntax**

```python
import math

root = math.isqrt(144)
combos = math.comb(10, 3)
```

**Öne Çıkan Detaylar**

* **Tam Sayı Matematiği Ayrımı:** Kayan noktalı (`float`) dönüşüm yapmadan doğrudan büyük tam sayılar üzerinde çalışan `gcd`, `lcm`, `isqrt`, `comb`, `perm` ve `factorial` gibi fonksiyonlar yüksek hassasiyetle çalışır.
* **Kapsamlı Hata Tanıları:** Yanlış girinti, kapanmamış parantezler, `TypedDict` / `dataclass` alan uyuşmazlıkları ve modül öznitelik hatalarında daha hedefli konum işaretçileri sunulur.


In [ ]:
import math

big_num: int = 10**30 + 7
root_val: int = math.isqrt(big_num)
gcd_val: int = math.gcd(48, 180)
lcm_val: int = math.lcm(12, 18)

print('isqrt(10**30 + 7):', root_val)
print('gcd(48, 180):', gcd_val)
print('lcm(12, 18):', lcm_val)
